# Day 4: Layers and Loss Functions (CS231n notes)

## Q1. What are logits, and how does softmax turn them into probabilities?

Logits are the raw, unnormalized numerical scores (they can range from negative to positive infinity) that the final layer of a neural network gives out. They show the model's unscaled confidence for each class or vocabulary token.

The softmax function turns a vector of logits $z$ into a clean probability distribution in two steps.

**Exponentiation ($e^{z_i}$):**
- I raise the constant $e$ to the power of each logit.
- This turns all negative numbers into positive values and amplifies the larger scores a lot compared to the smaller ones (this is the "soft-max" effect).

**Normalization ($\sum_j e^{z_j}$):**
- I add up the exponentiated values of all the classes.
- I divide each class's exponentiated score by that total sum.

**Formula.** For any class $i$ out of $K$ classes in total, the softmax probability $p_i$ is:

$$p_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}$$

The probability $p_i$ is calculated separately for each element $z_i$ of the logit vector, but the whole vector is used to calculate the shared denominator.

---

## Q2. Why is the loss $-\log(p_{correct})$?

This is called the Categorical Cross-Entropy loss (or Negative Log-Likelihood). It is used because it measures how "wrong" the model's probability distribution is compared to the actual truth.

A perfect model gives a probability of 1.0 (100%) to the correct class, and a terrible model gives 0.0 (0%). The negative log maps these probabilities to a penalty:
- If $p_{correct} = 1.0$, then $-\log(1.0) = \mathbf{0}$, so there is zero penalty for getting it perfectly right.
- If $p_{correct} = 0.5$, then $-\log(0.5) \approx \mathbf{0.69}$, so there is a moderate penalty for being unsure.
- If $p_{correct} \rightarrow 0.0$, then $-\log(0.0) \rightarrow \mathbf{\infty}$, so there is an infinite penalty for being completely wrong.

For a batch of $N$ examples, the loss is the average over all of them:

$$L = \frac{1}{N}\sum_{i=1}^{N} -\log\left(p_{i,\,correct}\right)$$

---

## Q3. Why does a random 10-class network start at a loss of about $\ln(10) \approx 2.30$?

At initialization the model has no knowledge, so it treats all classes as equally likely.

1. **Equal logits:** When a network is randomly initialized, its weights are small random numbers near zero. So the output logits for all 10 classes are roughly equal, e.g. $z \approx [0, 0, \dots, 0]$.
2. **Uniform probabilities:** Passing equal logits through softmax spreads the probability evenly. For $K$ classes, the probability of any single class is
$$p = \frac{1}{K}$$
For a 10-class network, the probability given to the correct class is $0.10$ (10%).
3. **The loss calculation:** Putting this 10% guess into the cross-entropy formula:
$$\text{Loss} = -\ln(p_{correct}) = -\ln\left(\frac{1}{10}\right) = \ln(10) \approx \mathbf{2.302}$$

I also checked this in code: 1000 random examples with logits scaled by 0.01 gave a loss of **2.30238**, and $\ln(10) = 2.30259$. If a new network starts far from 2.30, something is wrong in my code.

---

## Q4. Why does `exp()` overflow on big logits, and why does subtracting the max fix it without changing the answer?

Computers store decimals in the 32-bit floating-point format (float32). It has a hard ceiling: the biggest number it can store is about $3.4 \times 10^{38}$.

The exponential function $e^{x}$ grows so fast that it hits this limit early:
- $e^{88} \approx 1.6 \times 10^{38}$ (safe)
- $e^{89} \approx 4.4 \times 10^{38}$ (overflow, it goes over the maximum limit)

If a network gives a perfectly reasonable logit like $90.0$, computing $e^{90}$ gives `inf` (infinity), and then `inf/inf` gives `NaN` (Not a Number). This crashes the whole training.

(NumPy uses float64 by default, whose limit is much higher: $e^{709}$ is fine and $e^{710}$ is `inf`. In my test, the logits $[1000, 1001, 1002]$ gave `[nan nan nan]` with the naive softmax.)

**Why subtracting the max fixes it**

I find the maximum logit in the vector, $M = \max(z)$, and subtract it from every logit before running softmax. This moves the whole vector down so that the largest value becomes exactly $0$.
- The largest term becomes $e^0 = 1$.
- All the other terms become negative numbers, and $e^{\text{negative}}$ is a fraction between 0 and 1, so nothing can overflow.

**Why the answer does not change**

Subtracting $M$ from every logit is the same as multiplying the top and the bottom of the fraction by the same number $e^{-M}$, so it cancels out:

$$\frac{e^{z_i - M}}{\sum_j e^{z_j - M}} = \frac{e^{z_i}\,e^{-M}}{e^{-M}\sum_j e^{z_j}} = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

With the stable version, $[1000, 1001, 1002]$ gave `[0.090, 0.245, 0.665]`, and for small inputs like $[1, 2, 3]$ the stable and naive versions gave the same result.

---

## Q5. Derivation on paper: the gradient of softmax cross-entropy with respect to the logits is $(p - y)/N$

**Setup.** For one example, the logits are $z$, the probabilities are $p = \text{softmax}(z)$, and the correct class is $c$. Let $y$ be the one-hot vector ($y_j = 1$ if $j = c$, otherwise $0$). Let $S = \sum_k e^{z_k}$.

**Step 1: write the loss in a simple form.**

$$L = -\log(p_c) = -\log\left(\frac{e^{z_c}}{S}\right) = -z_c + \log S$$

**Step 2: differentiate with respect to a logit $z_j$.**

$$\frac{\partial L}{\partial z_j} = -\frac{\partial z_c}{\partial z_j} + \frac{\partial \log S}{\partial z_j}$$

- The first part: $\dfrac{\partial z_c}{\partial z_j} = 1$ if $j = c$, otherwise $0$. That is exactly $y_j$.
- The second part, by the chain rule:
$$\frac{\partial \log S}{\partial z_j} = \frac{1}{S}\cdot\frac{\partial S}{\partial z_j} = \frac{1}{S}\cdot e^{z_j} = p_j$$

**Step 3: put them together.**

$$\boxed{\frac{\partial L}{\partial z_j} = p_j - y_j}$$

So the gradient is "what the network predicted" minus "the truth". For the correct class it is $p_c - 1$, which is negative, so that logit is pushed up. For every other class it is $p_j - 0$, which is positive, so those logits are pushed down.

**Step 4: the batch.** The loss over $N$ examples is the average $L = \frac{1}{N}\sum_i L_i$, and each example's logits only affect its own $L_i$. So each example's gradient is divided by $N$:

$$\boxed{\frac{\partial L}{\partial z} = \frac{p - y}{N}}$$

In code (with `onehot` as the one-hot matrix): `grad = (softmax(logits) - onehot) / N`.

**Numerical check.** I nudged every logit by $10^{-6}$ both ways and recomputed the loss (central difference):

```
max error with (p - y)/N : 1.5966350463969547e-10
max error without the /N : 0.6858395837728115
```

So the formula is right, and forgetting the $/N$ is a real mistake that the check catches.

---

## Q6. Why all-zero weights fail: every neuron computes the same thing and gets the same gradient

If every weight in a layer is exactly zero, I run into the **symmetry problem** (the network becomes "homogenized"). Because every neuron starts identical, they compute exactly the same output, get exactly the same feedback, and update in lockstep. So the big network effectively reduces to a single, expensive neuron.

A basic dense neuron calculates:

$$\text{Output} = \sigma(\mathbf{W}\cdot\mathbf{x} + b)$$

If the weight matrix $\mathbf{W}$ is all zeros:
- The dot product $\mathbf{W}\cdot\mathbf{x}$ is exactly zero for every neuron, so the input features are completely ignored.
- If the biases $b$ are also zero, every hidden neuron outputs exactly 0 (or a constant like 0.5 with a sigmoid).
- **The result:** every neuron in a hidden layer becomes a perfect clone and sends the same value to the next layer.

I checked this on a small 2-layer network: with all weights equal (0 or 0.5), all the gradient columns of the first layer came out identical. With all zeros the gradients were exactly 0, so nothing could be learned at all.

---

## Q7. Why too-small weights make the signal shrink layer by layer, and too-large weights make it blow up

This is called the **vanishing and exploding gradient problem** (signal propagation failure). A deep network is basically a long chain of repeated matrix multiplications. If the weights are not scaled properly, the signal either shrinks to nothing or explodes out of control.

**1. Too-small weights: the shrinking signal (vanishing)**

If the weights are too small (their magnitude or variance is well below 1, e.g. around 0.1), each layer scales the incoming signal down.
- Think of starting with 1.0 and multiplying by 0.5 again and again:
  - Layer 1: $1.0 \times 0.5 = 0.5$
  - Layer 2: $0.5 \times 0.5 = 0.25$
  - Layer 5: $\approx 0.03$
  - Layer 50: $\approx 8.8 \times 10^{-16}$ (practically 0)
- **Impact:** In a deep 50-layer network the signal shrinks exponentially at every step, and by the final layers the activations are almost zero.
- **Backward problem:** Since the forward signals are about zero, the gradients in backprop also shrink to zero. The early layers get no feedback and stop learning.

**2. Too-large weights: the blowing-up signal (exploding)**

If the weights are too large (magnitude significantly greater than 1, e.g. around 2.0 or 3.0), each layer magnifies the signal.
- Think of starting with 1.0 and multiplying by 2.0 again and again:
  - Layer 1: $1.0 \times 2.0 = 2.0$
  - Layer 2: $2.0 \times 2.0 = 4.0$
  - Layer 5: $= 32.0$
  - Layer 50: $\approx 1.1 \times 10^{15}$ (an astronomical number)
- **Impact:** The activations explode exponentially, and soon go past what float32 can store, which gives `NaN` or `inf`.
- **Backward problem:** The gradients explode too. The optimizer then takes huge, chaotic steps that destroy the progress, and the loss jumps around wildly or breaks.

**My experiment** (10 layers, width 500, I printed the standard deviation of the final output):

| init | ReLU | tanh |
|---|---|---|
| tiny (0.01) | 7.5e-09 | 3.0e-07 |
| big (1.0) | 6.9e+11 | 0.98 (saturated near ±1) |
| Xavier $\sqrt{1/n}$ | 0.028 | 0.23 |
| He $\sqrt{2/n}$ | 0.81 | 0.55 |

Tiny weights make the signal vanish, and big weights make it explode (or saturate, for tanh).

---

## Q8. Xavier and He initialization

- **Xavier init** uses variance $\dfrac{1}{n_{in}}$. It is used for tanh and sigmoid.
- **He init** uses variance $\dfrac{2}{n_{in}}$. It is used for ReLU.

ReLU makes about half of the values zero, so I need the factor of 2 to keep the signal size steady. In the table above, with ReLU the Xavier version shrinks to 0.028 after 10 layers, while He stays at 0.81.

For my Linear layer the weight shape is `(in, out)`, so:

$$W = \text{randn}(in,\, out)\times\sqrt{\frac{2}{in}}$$

Note: the CS231n PDF on page 22 writes the ReLU rule as `/ sqrt(2n)`. That is a typo. The correct form is `* sqrt(2/n)`, which is also what its Summary on page 23 says.

**Biases:** I start them at 0, because the random weights already break the symmetry.

**Dead ReLU:** When I tried a very negative bias ($-5$) on a ReLU layer, 100% of the units were exactly 0 for every input. They get no gradient, so they can never recover.

---

## What I should be able to explain without notes
- Why softmax needs the max subtracted, and why it does not change the answer.
- Why the starting loss of a 10-class network is $\ln(10) \approx 2.30$.
- Why the gradient is $(p - y)/N$, and where the $/N$ comes from.
- Why all-zero weights fail.
- Why He init has the factor 2.

In [21]:
#Here, for the answers and theory being mentioned above, the code is also being shown for practice
#Why softmax needs the max-subtraction trick

import numpy as np
z = np.array([1000., 1001., 1002.])
naive = np.exp(z) / np.exp(z).sum()
print("naive:", naive)                       # nan, because exp(1000) overflows

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)    # subtract the max
    e = np.exp(z)
    return e / e.sum(axis=-1, keepdims=True)

print("stable:", softmax(z))
print("same answer on small inputs:", np.allclose(softmax(np.array([1.,2.,3.])),
      np.exp([1.,2.,3.])/np.exp([1.,2.,3.]).sum()))

naive: [nan nan nan]
stable: [0.09003057 0.24472847 0.66524096]
same answer on small inputs: True


/tmp/ipykernel_895/988747865.py:6: RuntimeWarning: overflow encountered in exp
  naive = np.exp(z) / np.exp(z).sum()
/tmp/ipykernel_895/988747865.py:6: RuntimeWarning: invalid value encountered in divide
  naive = np.exp(z) / np.exp(z).sum()


In [22]:
#Cross-entropy and the ln(10) fact

#At the start, every class gets about 10% probability, so the loss is about 2.30. This is the reason for having the value of loss close to - 2.30. If the real network starts far from that then, something is wrong.

rng = np.random.default_rng(0)
N, C = 1000, 10
y = rng.integers(0, C, size=N)
logits = rng.normal(size=(N, C)) * 0.01       # a network that knows nothing
p = softmax(logits)
loss = -np.log(p[np.arange(N), y]).mean()
print(loss, "vs ln(10) =", np.log(10))

2.30238079944263 vs ln(10) = 2.302585092994046


In [23]:
# Check the derivation numerically - This is the numberical analysis also done on paper for the verification: -

def ce(logits, y):
    p = softmax(logits)
    return -np.log(p[np.arange(len(y)), y]).mean()

N, C = 5, 4
logits = rng.normal(size=(N, C)); y = rng.integers(0, C, size=N)
onehot = np.eye(C)[y]
analytic = (softmax(logits) - onehot)/ N      # Mine derivation

num = np.zeros_like(logits); h = 1e-6
for i in range(N):
    for j in range(C):
        a = logits.copy(); a[i, j] += h
        b = logits.copy(); b[i, j] -= h
        num[i, j] = (ce(a, y) - ce(b, y)) / (2*h)
print("max error:", np.abs(analytic - num).max())   # about 1e-10

max error: 1.843645600768662e-10


In [24]:
# Same check as before, but with the /N removed: the error should be big
print("max error without the /N:", np.abs(analytic * N - num).max())

max error without the /N: 0.7556944100733555


In [25]:
# Weight initialization: watch the signal
# Idea: push random data through 10 layers and see if the signal stays healthy, shrinks to 0, or explodes.

def run(init, act, depth=10, width=500):
# Define a function run(). init = a function that gives the weight scale, act = the activation function, depth = number of layers, width = neurons per layer.

    x = rng.normal(size=(1000, width))
    # Make the input: 1000 examples, each with 500 random numbers from a normal distribution (mean 0, std 1). So the starting std is about 1.

    for _ in range(depth):
    # Repeat the next 3 lines 10 times, once per layer. "_" means I don't need the loop counter.

        W = rng.normal(size=(width, width)) * init(width)
        # Make this layer's weight matrix of shape (500, 500) from a normal distribution, then multiply by the scale init(500). This scale is the only thing that changes between the four experiments.

        x = act(x @ W)
        # Do the layer: x @ W is the matrix multiplication (1000x500 times 500x500 gives 1000x500), then apply the activation function. The result becomes the input of the next layer.

    return x.std()
    # After all 10 layers, return the standard deviation of the final output: one number that tells how big the signal is. Near 1 = healthy, near 0 = vanished, huge = exploded.

relu = lambda x: np.maximum(x, 0)
# Define ReLU as a small function: for every element take the larger of (x, 0), so negative values become 0 and positive values stay the same.

for name, f in [("tiny 0.01", lambda n: 0.01),
                ("big 1.0", lambda n: 1.0),
                ("Xavier sqrt(1/n)", lambda n: np.sqrt(1/n)),
                ("He sqrt(2/n)", lambda n: np.sqrt(2/n))]:
# Loop over 4 (label, scale function) pairs. Each scale function takes n (the number of inputs of the layer):
#   tiny   -> always 0.01
#   big    -> always 1.0
#   Xavier -> sqrt(1/n), which makes the weight variance 1/n
#   He     -> sqrt(2/n), which makes the weight variance 2/n

    print(f"{name:18s} std after 10 ReLU layers: {run(f, relu):.4g}")
    # Run the 10-layer experiment with this scale function and ReLU, then print the label (padded to 18 characters so the output lines up) and the final std (.4g = 4 significant digits).

tiny 0.01          std after 10 ReLU layers: 7.493e-09
big 1.0            std after 10 ReLU layers: 9.037e+11
Xavier sqrt(1/n)   std after 10 ReLU layers: 0.02199
He sqrt(2/n)       std after 10 ReLU layers: 0.6995


In [26]:
# The tanh column of the init table

for name, f in [("tiny 0.01", lambda n: 0.01), ("big 1.0", lambda n: 1.0),
                ("Xavier sqrt(1/n)", lambda n: np.sqrt(1/n)), ("He sqrt(2/n)", lambda n: np.sqrt(2/n))]:
    print(f"{name:18s} std after 10 tanh layers: {run(f, np.tanh):.4g}")

tiny 0.01          std after 10 tanh layers: 3.006e-07
big 1.0            std after 10 tanh layers: 0.9818
Xavier sqrt(1/n)   std after 10 tanh layers: 0.228
He sqrt(2/n)       std after 10 tanh layers: 0.554


In [27]:
# Cross-check with PyTorch

import torch, torch.nn.functional as F
lt = torch.tensor(logits, dtype=torch.float64, requires_grad=True)
F.cross_entropy(lt, torch.tensor(y)).backward()
print("torch loss:", F.cross_entropy(lt, torch.tensor(y)).item(), " mine:", ce(logits, y)) #This ce is coming from the one that has been defined by me earlier for the sake of checking the amount of error being calculated on paper.
print("grad match:", np.allclose(lt.grad.numpy(), analytic))

torch loss: 2.0577515123963304  mine: 2.0577515123963304
grad match: True


In [28]:
# Dead ReLU, in a few lines
# A ReLU unit that outputs 0 for every input gets zero gradient and stops learning. This is why initialization and biases matter.
# The learning got stopped after the first layer only.

x = rng.normal(size=(1000, 100)); W = rng.normal(size=(100, 100)) * 0.1
b = -5.0                                    # a very negative bias
out = np.maximum(x @ W + b, 0)
print("fraction of units that are exactly 0:", (out == 0).mean())

fraction of units that are exactly 0: 1.0


In [29]:
# The all-equal-weights symmetry demo

# 2-layer tanh network: every first-layer neuron gets the same gradient when all weights are equal
X = rng.normal(size=(8, 3)); yt = rng.normal(size=(8, 1))
for name, v in [("all zero", 0.0), ("all 0.5", 0.5)]:
    W1 = np.full((3, 4), v); W2 = np.full((4, 1), v)
    h = np.tanh(X @ W1); out = h @ W2; d = (out - yt) / len(X)
    dW2 = h.T @ d; dh = d @ W2.T * (1 - h**2); dW1 = X.T @ dh
    print(name, "| all columns of dW1 identical:", np.allclose(dW1, dW1[:, [0]]),
          "| max|dW1|:", abs(dW1).max(), "| max|dW2|:", abs(dW2).max())

all zero | all columns of dW1 identical: True | max|dW1|: 0.0 | max|dW2|: 0.0
all 0.5 | all columns of dW1 identical: True | max|dW1|: 0.35306026176789373 | max|dW2|: 0.7958041123330675
